In [104]:
import os, json, math, warnings, random
from typing import Dict, List, Tuple, Optional, Callable
from collections import defaultdict, Counter

import numpy as np
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torchvision.models as models

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [105]:
def presence_f1_from_logits(logits_p: torch.Tensor, gt_presence: torch.Tensor, thresh: float = 0.5, eps: float = 1e-6) -> float:
    """Macro F1 over part presence channels.
    logits_p: (B, P)
    gt_presence: (B, P) binary {0,1}
    """
    with torch.no_grad():
        prob = torch.sigmoid(logits_p)
        pred = (prob >= thresh).float()
        tp = (pred * gt_presence).sum(dim=0)
        fp = (pred * (1 - gt_presence)).sum(dim=0)
        fn = ((1 - pred) * gt_presence).sum(dim=0)
        precision = tp / (tp + fp + eps)
        recall    = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        # macro over parts (only parts that appear in dataset? we macro over all channels for stability)
        return float(f1.mean().item())

In [106]:
PART_VOCAB = [
    "head","body","foot","hand","tail","wing","fin",
    "engine","tire","seat","sail","side_mirror","mouth"
]
PART_TO_IDX = {p:i for i,p in enumerate(PART_VOCAB)}

# Regex-like matching (string contains) to map category names → canonical types
# tolerant to typos like "Tier" for "Tire".
CANON_RULES = [
    ("side mirror", "side_mirror"),
    ("tier", "tire"), ("tire", "tire"), ("tyre", "tire"),
    ("engine", "engine"),
    ("sail", "sail"),
    ("mouth", "mouth"),
    ("seat", "seat"),
    ("wing", "wing"),
    ("fin", "fin"),
    ("head", "head"),
    ("body", "body"),
    ("hand", "hand"),
    ("foot", "foot"),
    ("tail", "tail"),
]

def match_part_type(name: str) -> Optional[str]:
    n = name.lower()
    for token, canon in CANON_RULES:
        if token in n:
            return canon
    return None

# Weak prior for presence when JSON lacks pixel parts but has supercategory
SUPER_TO_PARTS = {
    # animals
    "Quadruped": {"head","body","foot","tail"},
    "Biped": {"head","body","hand","foot","tail"},  # tail appears in your list
    "Bird": {"head","body","wing","foot","tail"},
    "Fish": {"head","body","fin","tail","mouth"},
    "Snake": {"head","body","tail"},
    "Reptile": {"head","body","foot","tail"},
    # artifacts / vehicles
    "Car": {"body","tire","side_mirror"},
    "Bicycle": {"body","head","seat","tire"},
    "Boat": {"body","sail"},
    "Aeroplane": {"head","body","engine","wing","tail"},
    "Bottle": {"mouth","body"},
}

In [107]:
class PartImageNetCOCODataset(Dataset):
    """
    Expects COCO-style JSON where each annotation corresponds to a *part instance*.
    The specific part is given by `category_id`, and the categories array
    contains entries like {id, name: "Quadruped Head", supercategory: "Quadruped"}.

    We build a shared part space (channels) by mapping category *names* to
    canonical types (head/body/wing/...). If the JSON actually contains object
    categories (e.g., ImageNet synsets) and no part tokens, we fallback to
    weak presence labels derived from supercategory.
    """
    def __init__(self,
                 ann_path: str,
                 img_root: str,
                 img_size: Tuple[int,int]=(224,224),
                 augment: bool=True,
                 shared_by_type: bool=True):
        super().__init__()
        self.ann_path = ann_path
        self.img_root = img_root
        self.W, self.H = img_size[0], img_size[1]
        self.augment = augment
        self.shared_by_type = shared_by_type

        with open(ann_path, 'r') as f:
            data = json.load(f)

        # Index images, annotations, categories
        self.images: Dict[int,dict] = {im['id']: im for im in data['images']}
        self.img_ids: List[int] = list(self.images.keys())
        self.categories: Dict[int,dict] = {c['id']: c for c in data['categories']}

        self.anns_by_img: Dict[int,List[dict]] = defaultdict(list)
        for a in data['annotations']:
            self.anns_by_img[a['image_id']].append(a)

        # Build label space (supercategories)
        self.supercats = sorted({c.get('supercategory', 'unknown') for c in self.categories.values()})
        self.super_to_idx = {s:i for i,s in enumerate(self.supercats)}

        # Build shared part-type channels from category *names*
        part_types = []
        for c in self.categories.values():
            ptype = match_part_type(c['name'])
            if ptype and ptype not in part_types:
                part_types.append(ptype)
        part_types.sort()
        self.part_types = part_types  # actual seen types in this split
        self.num_part_channels = len(part_types) if shared_by_type else len(self.categories)

        # Map category_id → channel index
        self.channel_of_cat: Dict[int, int] = {}
        if shared_by_type:
            for cid, c in self.categories.items():
                ptype = match_part_type(c['name'])
                if ptype is not None and ptype in self.part_types:
                    self.channel_of_cat[cid] = self.part_types.index(ptype)
        else:
            for cid in self.categories:
                self.channel_of_cat[cid] = cid  # one channel per category

        # Heuristic: detect if *any* part polygons are likely present
        # If none of the category names matched part tokens, treat as object-only
        self.has_part_tokens = (len(self.part_types) > 0)

        print(f"[init] ann_path={ann_path}")
        print(f"[init] images={len(self.img_ids)} | categories={len(self.categories)}")
        print(f"[init] Recognized part types (shared): {self.part_types}")
        print(f"[init] Shared-by-type part channels: {self.num_part_channels}; Total input channels: {3 + self.num_part_channels}")

        # Transforms
        base = [T.Resize((self.H, self.W))]
        if augment:
            base += [T.RandomHorizontalFlip(p=0.5), T.RandomRotation(degrees=10)]
        base += [T.ToTensor()]
        self.img_tf = T.Compose(base)

    def __len__(self):
        return len(self.img_ids)

    def _image_path(self, info: dict) -> str:
        # JSON typically stores file_name relative to split root, e.g. "n014xxx/JPEG..."
        return os.path.join(self.img_root, info['file_name'])

    def _draw_polygon(self, draw: ImageDraw.ImageDraw, poly: list, info: dict):
        # `poly` is a flat list [x0,y0, x1,y1, ...] in original image coordinates
        xs, ys = poly[0::2], poly[1::2]
        ow, oh = info['width'], info['height']
        pts = []
        for x, y in zip(xs, ys):
            xi = int(round(x / ow * self.W))
            yi = int(round(y / oh * self.H))
            pts.append((xi, yi))
        if len(pts) >= 3:
            draw.polygon(pts, outline=1, fill=1)

    def _rasterize_masks(self, iid: int) -> torch.Tensor:
        """Return (C,H,W) mask tensor in {0,1}. If no recognized part tokens,
        returns zeros (we'll use weak presence by supercategory).
        """
        C = self.num_part_channels
        mask_imgs = [Image.new('L', (self.W, self.H), 0) for _ in range(C)]
        draws = [ImageDraw.Draw(mi) for mi in mask_imgs]

        anns = self.anns_by_img.get(iid, [])
        if len(anns) == 0 or C == 0:
            return torch.zeros((C, self.H, self.W), dtype=torch.float32)

        for ann in anns:
            cid = int(ann['category_id'])
            ch = self.channel_of_cat.get(cid)
            if ch is None:
                continue
            seg = ann.get('segmentation', None)
            if isinstance(seg, list) and len(seg) > 0:
                for poly in seg:
                    if isinstance(poly, list) and len(poly) >= 6:
                        self._draw_polygon(draws[ch], poly, self.images[iid])
            else:
                # bbox fallback if segmentation missing
                x, y, w, h = ann['bbox']
                info = self.images[iid]
                ow, oh = info['width'], info['height']
                xs = int(round(x / ow * self.W))
                ys = int(round(y / oh * self.H))
                xe = int(round((x + w) / ow * self.W))
                ye = int(round((y + h) / oh * self.H))
                draws[ch].rectangle([xs, ys, xe, ye], outline=1, fill=1)

        mask = torch.stack([TF.to_tensor(mi).squeeze(0) for mi in mask_imgs], dim=0).to(torch.float32)
        return mask

    def __getitem__(self, idx: int):
        iid = self.img_ids[idx]
        info = self.images[iid]
        img_path = self._image_path(info)
        if not os.path.exists(img_path):
            # Safe fail: return None, collate will drop
            return None

        img = Image.open(img_path).convert('RGB')
        img = self.img_tf(img)  # (3,H,W)

        # Label (supercategory of *first* annotation)
        anns = self.anns_by_img.get(iid, [])
        if len(anns) == 0:
            label_idx = 0
            super_name = self.supercats[0]
        else:
            first_cid = int(anns[0]['category_id'])
            super_name = self.categories[first_cid].get('supercategory', 'unknown')
            label_idx = self.super_to_idx.get(super_name, 0)

        # Pixel masks (shared)
        mask = self._rasterize_masks(iid) if self.has_part_tokens else torch.zeros((self.num_part_channels, self.H, self.W))

        if self.augment:
            do_flip = random.random() < 0.5
            angle = random.uniform(-10, 10)
            if do_flip:
                img = TF.hflip(img)
                mask = torch.flip(mask, dims=[2])  # flip width dim: (C,H,W) -> dim=2
            if angle != 0:
                img  = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
                # rotate mask per-channel with NEAREST to preserve 0/1
                mask = TF.rotate(
                    mask.unsqueeze(0),              # (1, C, H, W) -> 4D so grid_sample does 2D case
                    angle,
                    interpolation=TF.InterpolationMode.NEAREST,
                    fill=0
                ).squeeze(0)                        # back to (C, H, W)
        
        img  = TF.resize(img, (self.H, self.W))
        # Ensure `img` is a tensor float in [0,1] regardless of input type
        if isinstance(img, Image.Image):
            img = TF.pil_to_tensor(img)                 # uint8 [0..255], (C,H,W)
        # if it's already a Tensor, keep as-is, just convert dtype/scale
        img = TF.convert_image_dtype(img, torch.float32)  # -> float32 [0..1]

        # Part presence target
        if self.has_part_tokens:
            presence = (mask.sum(dim=(1,2)) > 0).float()
            has_pix_flag = float(presence.sum().item() > 0)
        else:
            presence = torch.zeros(len(PART_VOCAB), dtype=torch.float32)
            for p in SUPER_TO_PARTS.get(super_name, set()):
                presence[PART_TO_IDX[p]] = 1.0
            # If dataset shares only the recognized part_types subset, crop presence
            if self.num_part_channels != len(PART_VOCAB):
                # Build mapping vector from PART_VOCAB → part_types subset
                pres_sub = torch.zeros(self.num_part_channels, dtype=torch.float32)
                for i, p in enumerate(self.part_types):
                    pres_sub[i] = presence[PART_TO_IDX[p]]
                presence = pres_sub
            has_pix_flag = 0.0

        return img, torch.tensor(label_idx, dtype=torch.long), mask, presence, torch.tensor(has_pix_flag, dtype=torch.float32)


In [108]:
def collate_drop_none(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    imgs, labels, masks, presence, flags = zip(*batch)
    return (
        torch.stack(imgs, dim=0),
        torch.stack(labels, dim=0),
        torch.stack(masks, dim=0),
        torch.stack(presence, dim=0),
        torch.stack(flags, dim=0),
    )



In [109]:

class SharedMTLResNet(nn.Module):
    def __init__(self, num_labels: int, num_parts: int,
                 use_sam: bool=False, sam_predictor: Optional[Callable]=None):
        super().__init__()
        self.use_sam = use_sam
        self.sam_predictor = sam_predictor

        base = models.resnet18(weights=None)
        self.backbone = base

        # Replace first conv to accept 3 + num_parts channels
        in_ch = 3 + num_parts
        self.backbone.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)

        feat_dim = base.fc.in_features
        self.backbone.fc = nn.Identity()

        self.fc_label = nn.Linear(feat_dim, num_labels)
        self.fc_parts = nn.Linear(feat_dim, num_parts)   # part PRESENCE head (multi-label)

    def forward(self, x_rgb: torch.Tensor, x_masks: Optional[torch.Tensor]=None):
        if x_masks is None or x_masks.size(1) == 0:
            # Optional: call SAM to get pseudo part masks
            if self.use_sam and self.sam_predictor is not None:
                with torch.no_grad():
                    x_masks = self.sam_predictor(x_rgb)
            else:
                x_masks = torch.zeros(x_rgb.size(0), 0, x_rgb.size(2), x_rgb.size(3), device=x_rgb.device)
        x = torch.cat([x_rgb, x_masks], dim=1)
        feats = self.backbone(x)
        logits_y = self.fc_label(feats)
        logits_p = self.fc_parts(feats)
        return logits_y, logits_p

In [110]:
def part_presence_f1(prob: torch.Tensor, target: torch.Tensor, thresh: float=0.5) -> float:
    pred = (prob > thresh).int()
    tgt = target.int()
    tp = (pred & tgt).sum().item()
    fp = (pred & (1 - tgt)).sum().item()
    fn = ((1 - pred) & tgt).sum().item()
    if tp + fp + fn == 0:
        return 0.0
    return 2 * tp / (2 * tp + fp + fn)

class EarlyStopper:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = -1e9
        self.count = 0
        self.stop = False
    def step(self, metric: float):
        if metric > self.best + self.min_delta:
            self.best = metric
            self.count = 0
        else:
            self.count += 1
            if self.count >= self.patience:
                self.stop = True

In [111]:
def train_one_epoch(model, loader, opt):
    model.train()
    tot, corr = 0, 0
    f1s = []
    for step, batch in enumerate(loader):
        if batch is None:
            continue
        x, y, m, pres, has_pix = batch
        x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)

        logits_y, logits_p = model(x, m)
        loss_y = F.cross_entropy(logits_y, y)
        loss_p = F.binary_cross_entropy_with_logits(logits_p, pres)
        loss = loss_y + 0.1 * loss_p

        opt.zero_grad(); loss.backward(); opt.step()

        corr += (logits_y.argmax(1) == y).sum().item()
        tot += y.size(0)
        f1s.append(part_presence_f1(torch.sigmoid(logits_p).detach(), pres))

        if step % 100 == 0:
            # Diagnostics: how many channels are entirely zero in this batch's GT masks
            zero_ch = int((m.sum(dim=(0,2,3)) == 0).sum().item()) if m.numel() > 0 else 0
            print(f"[dbg] step {step} | gt zero-channels (count over batch*channels): {zero_ch}")

    acc = corr / max(1, tot)
    f1 = float(np.mean(f1s)) if len(f1s) else 0.0
    return acc, f1

def evaluate(model, loader):
    model.eval()
    tot, corr = 0, 0
    f1s = []
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            x, y, m, pres, has_pix = batch
            x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
            logits_y, logits_p = model(x, m)
            corr += (logits_y.argmax(1) == y).sum().item()
            tot += y.size(0)
            f1s.append(part_presence_f1(torch.sigmoid(logits_p), pres))
    acc = corr / max(1, tot)
    f1 = float(np.mean(f1s)) if len(f1s) else 0.0
    return acc, f1

In [112]:
def audit_masks(ds: PartImageNetCOCODataset, max_images=200):
    n = min(max_images, len(ds))
    nz_images = 0
    nz_channels = set()
    for i in range(n):
        sample = ds[i]
        if sample is None:
            continue
        _, _, masks, presence, has_pix = sample
        if (masks.sum() > 0).item():
            nz_images += 1
        for ch in range(masks.size(0)):
            if masks[ch].sum().item() > 0:
                nz_channels.add(ch)
    print(f"[audit] images scanned: {n}")
    print(f"[audit] images with any foreground mask: {nz_images}")
    print(f"[audit] approx. part-channels with any pixels across scanned images: {len(nz_channels)}")


In [113]:
ROOT = "PartImageNet_Seg/PartImageNet"

TRAIN_JSON     = os.path.join(ROOT, "annotations/train/train.json")
VAL_JSON       = os.path.join(ROOT, "annotations/val/val.json")
TRAIN_IMG_ROOT = os.path.join(ROOT, "images/train")
VAL_IMG_ROOT   = os.path.join(ROOT, "images/val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = PartImageNetCOCODataset(
    ann_path=TRAIN_JSON,
    img_root=TRAIN_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=True,
    shared_by_type=True,
)
val_ds = PartImageNetCOCODataset(
    ann_path=VAL_JSON,
    img_root=VAL_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=False,
    shared_by_type=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          collate_fn=collate_drop_none, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                          collate_fn=collate_drop_none, drop_last=False)

print("\nRunning audit on training data (first 200 images)…")
audit_masks(train_ds, max_images=200)

[init] ann_path=PartImageNet_Seg/PartImageNet/annotations/train/train.json
[init] images=20481 | categories=40
[init] Recognized part types (shared): ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
[init] Shared-by-type part channels: 13; Total input channels: 16
[init] ann_path=PartImageNet_Seg/PartImageNet/annotations/val/val.json
[init] images=1206 | categories=40
[init] Recognized part types (shared): ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
[init] Shared-by-type part channels: 13; Total input channels: 16

Running audit on training data (first 200 images)…
[audit] images scanned: 200
[audit] images with any foreground mask: 200
[audit] approx. part-channels with any pixels across scanned images: 3


## Independent Channel

In [114]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

@dataclass
class IndepSpec:
    """Compact independent-channel spec (K_max = max parts per class).
    For each class c, slot k uses shared channel `table[c,k]` or -1 (blank).
    """
    supers: List[str]                      # ordered superclass names
    part_to_shared: Dict[str, int]         # global part name -> shared idx (0..K_shared-1)
    class2len: Dict[str, int]              # number of parts this class has (<= K_max)
    table: torch.LongTensor                # shape (C, K_max), entries in [-1..K_shared-1]

    @property
    def K_shared(self) -> int:
        return int(max(self.part_to_shared.values()) + 1) if self.part_to_shared else 0

    @property
    def K_max(self) -> int:
        return int(self.table.shape[1]) if isinstance(self.table, torch.Tensor) else 0

from collections.abc import Sequence

def build_indep_spec(supercats: list[str]) -> IndepSpec:
    """
    Build compact spec with K_max channels (K_max = max parts across classes),
    robust to SUPER_TO_PARTS entries being sets, lists, tuples, or having duplicates.

    Expects globals: PART_VOCAB, PART_TO_IDX, SUPER_TO_PARTS
    """
    assert 'PART_VOCAB' in globals() and 'SUPER_TO_PARTS' in globals(), \
        "PART_VOCAB and SUPER_TO_PARTS must be defined"
    # Allow PART_TO_IDX to be absent; derive if needed
    part_to_shared = {p: (int(PART_TO_IDX[p]) if 'PART_TO_IDX' in globals() else i)
                      for i, p in enumerate(PART_VOCAB)}

    # Normalize per-class part lists (ordered, de-duped) and validate
    class_parts: dict[str, list[str]] = {}
    for s in supercats:
        raw = SUPER_TO_PARTS.get(s, [])
        if isinstance(raw, Sequence) and not isinstance(raw, (str, bytes)):
            # keep original order, drop duplicates
            seen = set()
            parts = []
            for p in raw:
                if p in seen:
                    continue
                if p not in part_to_shared:
                    raise KeyError(f"Part '{p}' for class '{s}' not in PART_VOCAB")
                seen.add(p)
                parts.append(p)
        else:
            # set or other iterable: sort deterministically by global shared index
            parts = sorted(list(raw), key=lambda p: part_to_shared.get(p, 10**9))
            for p in parts:
                if p not in part_to_shared:
                    raise KeyError(f"Part '{p}' for class '{s}' not in PART_VOCAB")
        class_parts[s] = parts

    # Compute K_max after normalization
    K_max = max((len(v) for v in class_parts.values()), default=0)
    C = len(supercats)

    # Build the (C, K_max) table of shared indices; -1 marks blank slots
    table = torch.full((C, K_max), -1, dtype=torch.long)
    for c, s in enumerate(supercats):
        for k, p in enumerate(class_parts[s]):
            if k >= K_max:
                break
            table[c, k] = part_to_shared[p]

    class2len = {s: len(class_parts[s]) for s in supercats}
    return IndepSpec(supercats, part_to_shared, class2len, table)

@torch.no_grad()
def pack_shared_to_indep_batch(m_shared: torch.Tensor,
                               pres_shared: torch.Tensor,
                               super_idx: torch.Tensor,
                               spec,
                               super_list: List[str],
                               part_types=None):  # <- extra arg accepted & ignored
    """
    Compact mapping: (B, K_shared, H, W) -> (B, K_max, H, W) using class-dependent slots.
    Trailing channels are zero when a class has < K_max parts.
    Returns: (m_indep, y_indep)
    """
    B, K_shared, H, W = m_shared.shape
    assert K_shared == spec.K_shared, f"K_shared mismatch: {K_shared} vs {spec.K_shared}"

    # (B, K_max) indices of which shared channel fills each compact slot; -1 means blank
    idx = spec.table.index_select(0, super_idx.detach().cpu()).to(m_shared.device)
    valid = (idx >= 0).unsqueeze(-1).unsqueeze(-1)                     # (B, K_max, 1, 1)

    # Gather masks over channel dim; replace -1 with 0 for safe gather, then zero out invalid
    idx_safe = idx.clamp_min(0).unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)
    m_ind = m_shared.gather(1, idx_safe)                                # (B, K_max, H, W)
    m_ind = torch.where(valid, m_ind, torch.zeros_like(m_ind))

    # Presence targets mapped to compact slots
    y_ind = pres_shared.gather(1, idx.clamp_min(0))                     # (B, K_max)
    y_ind = torch.where(idx >= 0, y_ind, torch.zeros_like(y_ind))
    return m_ind, y_ind

@torch.no_grad()
def indep_logits_to_shared(logits_ind: torch.Tensor,
                           y: torch.Tensor,
                           spec,
                           super_list: List[str],
                           part_types=None):  # <- extra arg accepted & ignored
    """
    Map compact independent-slot logits back to shared-part logits per sample.
    Returns (B, K_shared).
    """
    B = logits_ind.size(0)
    out = torch.full((B, spec.K_shared), fill_value=-1e4, device=logits_ind.device)
    idx = spec.table.index_select(0, y.detach().cpu()).to(logits_ind.device)  # (B, K_max)
    valid = (idx >= 0)
    for b in range(B):
        if valid[b].any():
            out[b, idx[b][valid[b]]] = logits_ind[b][valid[b]]
    return out


class IndependentMTLResNet(nn.Module):
    """Multitask ResNet that consumes INDEPENDENT channels and predicts:
    - label logits over supercategories
    - part-presence logits over independent (super,part) tuples
    """
    def __init__(self, num_labels: int, num_indep_parts: int):
        super().__init__()
        base = models.resnet18(weights=None)
        in_ch = 3 + num_indep_parts
        base.conv1 = nn.Conv2d(in_ch, 64, kernel_size=7, stride=2, padding=3, bias=False)
        feat_dim = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base
        self.fc_label = nn.Linear(feat_dim, num_labels)
        self.fc_ind   = nn.Linear(feat_dim, num_indep_parts)

    def forward(self, x_rgb: torch.Tensor, x_mind: torch.Tensor):
        x = torch.cat([x_rgb, x_mind], dim=1)
        feats = self.backbone(x)
        return self.fc_label(feats), self.fc_ind(feats)

In [115]:
def train_one_epoch_indep(model: nn.Module, loader, spec: IndepSpec, super_list: List[str], opt) -> Tuple[float,float]:
    model.train(); tot, corr = 0, 0; f1s = []
    for step, (x, y, m_shared, pres_shared, haspix) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        m_shared, pres_shared = m_shared.to(device), pres_shared.to(device)
        m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, train_ds.supercats, train_ds.part_types)
        logits_y, logits_i = model(x, m_ind)
        loss_y = F.cross_entropy(logits_y, y)
        loss_i = F.binary_cross_entropy_with_logits(logits_i, y_ind)
        loss = loss_y + 0.1 * loss_i
        opt.zero_grad(); loss.backward(); opt.step()
        corr += (logits_y.argmax(1) == y).sum().item(); tot += y.size(0)
        f1s.append(presence_f1_from_logits(logits_i, y_ind))
        if step % 100 == 0:
            zeros = int((m_ind.sum(dim=(0,2,3)) == 0).sum().item())
            print(f"[indep] step {step} | zero indep-channels across batch: {zeros}/{m_ind.size(1)}")
    return corr/max(1,tot), float(np.mean(f1s))


def evaluate_indep(model: nn.Module, loader, spec: IndepSpec, super_list: List[str]) -> Tuple[float,float]:
    model.eval(); tot, corr = 0, 0; f1s = []
    with torch.no_grad():
        for (x, y, m_shared, pres_shared, haspix) in loader:
            x, y = x.to(device), y.to(device)
            m_shared, pres_shared = m_shared.to(device), pres_shared.to(device)
            m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, super_list, train_ds.part_types)
            logits_y, logits_i = model(x, m_ind)
            corr += (logits_y.argmax(1) == y).sum().item(); tot += y.size(0)
            f1s.append(presence_f1_from_logits(logits_i, y_ind))
    return corr/max(1,tot), float(np.mean(f1s))

## Base ResNet

In [116]:
import os, json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score
import numpy as np

In [117]:
class PartImageNetDatasetBS(Dataset):
    def __init__(self, root, ann_path, transform=None):
        self.root = root
        self.transform = transform
        with open(ann_path, "r") as f:
            self.ann = json.load(f)

        self.images = {}
        for obj in self.ann["images"]:
            self.images[obj["id"]] = obj["file_name"]

        self.parts = {}
        for ann in self.ann["annotations"]:
            img_id = ann["image_id"]
            cat_id = ann["category_id"]
            if img_id not in self.parts:
                self.parts[img_id] = []
            self.parts[img_id].append(cat_id)

        self.categories = self.ann["categories"]
        self.supercats = sorted(set([c["supercategory"] for c in self.categories]))
        self.super_to_idx = {s:i for i,s in enumerate(self.supercats)}

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_id = list(self.images.keys())[idx]
        img_path = os.path.join(self.root, self.images[img_id])
        image = Image.open(img_path).convert("RGB")

        label = self.super_to_idx[self.categories[self.parts[img_id][0]]["supercategory"]] \
                if img_id in self.parts else 0

        # Multi-hot part vector
        part_vec = np.zeros(len(self.categories), dtype=np.float32)
        if img_id in self.parts:
            for cid in self.parts[img_id]:
                part_vec[cid] = 1.0

        if self.transform:
            image = self.transform(image)

        return image, label, torch.tensor(part_vec)

In [118]:
class BaselineMTLResNet(nn.Module):
    def __init__(self, num_labels, num_parts):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc_label = nn.Linear(base.fc.in_features, num_labels)
        self.fc_part = nn.Linear(base.fc.in_features, num_parts)

    def forward(self, x):
        feats = self.backbone(x).flatten(1)
        out_label = self.fc_label(feats)
        out_part = torch.sigmoid(self.fc_part(feats))
        return out_label, out_part
    
def presence_f1(y_true, y_pred):
    y_true = y_true.cpu().numpy()
    y_pred = (y_pred.cpu().numpy() > 0.5).astype(int)
    return f1_score(y_true, y_pred, average="macro", zero_division=0)

In [119]:
def train_one_epoch(model, loader, opt, criterion_c, criterion_p):
    model.train()
    total, correct, total_loss = 0,0,0
    for imgs, labels, parts in loader:
        imgs, labels, parts = imgs.to(device), labels.to(device), parts.to(device)
        opt.zero_grad()
        out_c, out_p = model(imgs)
        loss_c = criterion_c(out_c, labels)
        loss_p = criterion_p(out_p, parts)
        loss = loss_c + 0.5*loss_p
        loss.backward()
        opt.step()
        total_loss += loss.item()
        total += labels.size(0)
        correct += (out_c.argmax(1) == labels).sum().item()
    return correct/total, total_loss/len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion_c, criterion_p):
    model.eval()
    total, correct, total_loss = 0,0,0
    all_true, all_pred = [], []
    for imgs, labels, parts in loader:
        imgs, labels, parts = imgs.to(device), labels.to(device), parts.to(device)
        out_c, out_p = model(imgs)
        loss_c = criterion_c(out_c, labels)
        loss_p = criterion_p(out_p, parts)
        loss = loss_c + 0.5*loss_p
        total_loss += loss.item()
        total += labels.size(0)
        correct += (out_c.argmax(1) == labels).sum().item()
        all_true.append(parts.cpu())
        all_pred.append(out_p.cpu())
    f1 = presence_f1(torch.cat(all_true), torch.cat(all_pred))
    return correct/total, total_loss/len(loader), f1

## Integrated Training

In [120]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from typing import List, Tuple, Dict, Optional
import copy

In [121]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

In [122]:
# Replace your current collate_fn with this one:
def collate_fn(batch):
    import torch
    good = []
    for item in batch:
        # each item should be: (img, label_idx, mask[C,H,W], presence[C], haspix_flag)
        if item is None: 
            continue
        if not isinstance(item, (tuple, list)) or len(item) != 5:
            continue
        x, y, m, pres, haspix = item
        if m is None or pres is None:
            continue
        # Per-item checks: mask is 3D [C,H,W], presence is 1D [C]
        if not hasattr(m, "ndim") or m.ndim != 3:
            continue
        if not hasattr(pres, "ndim") or pres.ndim != 1:
            continue
        if m.shape[0] != pres.shape[0]:
            continue
        good.append((x, y, m, pres, haspix))

    if not good:
        raise ValueError("All items in batch were invalid; let DataLoader resample.")

    xs   = torch.stack([g[0] for g in good], dim=0)                       # [B,3,H,W]
    ys   = torch.tensor([int(g[1]) for g in good], dtype=torch.long)      # [B]
    ms   = torch.stack([g[2] for g in good], dim=0)                       # [B,C,H,W] (C can be 0)
    pres = torch.stack([g[3] for g in good], dim=0).to(torch.float32)     # [B,C]
    hpix = torch.tensor([float(g[4]) for g in good], dtype=torch.float32) # [B]
    return xs, ys, ms, pres, hpix


In [123]:
@torch.no_grad()
def macro_presence_f1_from_logits_shared(logits_shared: torch.Tensor,
                                         pres_shared: torch.Tensor,
                                         thresh: float = 0.5,
                                         eps: float = 1e-6) -> float:
    """
    logits_shared: (B, P_shared) raw logits for shared parts
    pres_shared:   (B, P_shared) in {0,1}
    """
    prob = torch.sigmoid(logits_shared)
    pred = (prob >= thresh).float()

    # Per-channel F1, then macro
    tp = (pred * pres_shared).sum(dim=0)
    fp = (pred * (1 - pres_shared)).sum(dim=0)
    fn = ((1 - pred) * pres_shared).sum(dim=0)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    return float(f1.mean().item())

@torch.no_grad()
def acc_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    return float((logits.argmax(1) == y).float().mean().item())


In [124]:
@torch.no_grad()
def _set_bn_eval(m):
    if isinstance(m, torch.nn.modules.batchnorm._BatchNorm):
        m.eval()

@torch.no_grad()
def _macro_f1_from_logits(L, Y, thr=0.5, eps=1e-6):
    P = L.sigmoid()
    pred = (P >= thr).float()
    tp = (pred*Y).sum(0); fp = (pred*(1-Y)).sum(0); fn = ((1-pred)*Y).sum(0)
    f1 = (2*tp)/(2*tp + fp + fn + eps)
    return float(f1.mean().item())

In [125]:
def get_model(model_type, num_classes, num_parts):
    if model_type == "shared":
        return SharedMTLResNet(num_classes=num_classes, num_parts=num_parts)
    elif model_type == "independent":
        return IndependentMTLResNet(num_classes=num_classes, num_parts=num_parts)
    elif model_type == "baseline":
        return BaselineMTLResNet(num_classes=num_classes, num_parts=num_parts)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

In [126]:
def run_baseline(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, collate_fn=collate_fn)

    num_labels = len(train_ds.supercats)
    num_parts  = len(PART_VOCAB)
    model = BaselineMTLResNet(num_labels, num_parts).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    best = {'acc': 0.0, 'f1': 0.0}
    for ep in range(1, epochs+1):
        model.train()
        for x, y, _m, pres, _ in tl:
            x, y, pres = x.to(device), y.to(device), pres.to(device)
            logits_y, logits_p = model(x)
            loss = F.cross_entropy(logits_y, y) + 0.1 * F.binary_cross_entropy_with_logits(logits_p, pres)
            opt.zero_grad(); loss.backward(); opt.step()

        # eval
        model.eval(); accs, f1s = [], []
        with torch.no_grad():
            for x, y, _m, pres, _ in vl:
                x, y, pres = x.to(device), y.to(device), pres.to(device)
                logits_y, logits_p = model(x)
                accs.append(_acc_from_logits(logits_y, y))
                f1s.append(presence_f1(torch.sigmoid(logits_p), pres))
        acc, f1 = float(np.mean(accs)), float(np.mean(f1s))
        best['acc'] = max(best['acc'], acc); best['f1'] = max(best['f1'], f1)
        print(f"[BASE]  Epoch {ep:02d} | Val Acc {acc:6.2%} | Part F1 {f1:.3f}")
    return best

def run_shared(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, collate_fn=collate_fn)

    num_labels = len(train_ds.supercats)
    num_parts  = len(PART_VOCAB)
    model = SharedMTLResNet(num_labels, num_parts).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    best = {'acc': 0.0, 'f1': 0.0}
    for ep in range(1, epochs+1):
        model.train()
        for x, y, m, pres, _ in tl:
            x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
            logits_y, logits_p = model(x, m)
            loss = F.cross_entropy(logits_y, y) + 0.1 * F.binary_cross_entropy_with_logits(logits_p, pres)
            opt.zero_grad(); loss.backward(); opt.step()

        # eval
        model.eval(); accs, f1s = [], []
        with torch.no_grad():
            for x, y, m, pres, _ in vl:
                x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
                logits_y, logits_p = model(x, m)
                accs.append(_acc_from_logits(logits_y, y))
                f1s.append(presence_f1(torch.sigmoid(logits_p), pres))
        acc, f1 = float(np.mean(accs)), float(np.mean(f1s))
        best['acc'] = max(best['acc'], acc); best['f1'] = max(best['f1'], f1)
        print(f"[SHARED] Epoch {ep:02d} | Val Acc {acc:6.2%} | Part F1 {f1:.3f}")
    return best

def run_indep(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, collate_fn=collate_fn)

    num_labels = len(train_ds.supercats)
    spec = build_indep_spec(train_ds.supercats)
    K = spec.K_max
    model = IndependentMTLResNet(num_labels, K).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    best = {'acc': 0.0, 'f1': 0.0}
    for ep in range(1, epochs+1):
        model.train()
        for x, y, m, pres, _ in tl:
            x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
            m_ind, y_ind = pack_shared_to_indep_batch(m, pres, y, spec, train_ds.supercats, train_ds.part_types)
            logits_y, logits_i = model(x, m_ind)
            loss = F.cross_entropy(logits_y, y) + 0.1 * F.binary_cross_entropy_with_logits(logits_i, y_ind)
            opt.zero_grad(); loss.backward(); opt.step()

        # eval: map indep part logits back to SHARED space for presence-F1
        model.eval(); accs, f1s = [], []
        with torch.no_grad():
            for x, y, m, pres, _ in vl:
                x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device)
                m_ind, _ = pack_shared_to_indep_batch(m, pres, y, spec, val_ds.supercats, train_ds.part_types)
                logits_y, logits_i = model(x, m_ind)
                accs.append(_acc_from_logits(logits_y, y))
                logits_shared = indep_logits_to_shared(logits_i, y, spec, val_ds.supercats)
                f1s.append(presence_f1(torch.sigmoid(logits_shared), pres))
        acc, f1 = float(np.mean(accs)), float(np.mean(f1s))
        best['acc'] = max(best['acc'], acc); best['f1'] = max(best['f1'], f1)
        print(f"[INDEP]  Epoch {ep:02d} | Val Acc {acc:6.2%} | Part F1 {f1:.3f}")
    return best

In [127]:
def compare_all(epochs=3, lr=1e-3, bs=32, train_loader=None, val_loader=None):
    import pandas as pd

    print("=== Running Baseline (RGB-only) ===")
    res_base   = run_baseline(epochs, lr, bs, train_loader, val_loader)

    print("\n=== Running Shared-channels ===")
    res_shared = run_shared(epochs, lr, bs, train_loader, val_loader)

    print("\n=== Running Independent-channels ===")
    res_indep  = run_indep(epochs, lr, bs, train_loader, val_loader)

    summary = pd.DataFrame([
        {"Method": "Baseline (RGB)",      "Best Label Acc": res_base['acc'],   "Best Part F1": res_base['f1']},
        {"Method": "Shared-channels",      "Best Label Acc": res_shared['acc'], "Best Part F1": res_shared['f1']},
        {"Method": "Independent-channels", "Best Label Acc": res_indep['acc'],  "Best Part F1": res_indep['f1']},
    ])
    print("\n=== Summary ===")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return summary

In [128]:
@torch.no_grad()
def evaluate_shared(model, loader, device, thr=0.5):
    model.eval(); model.apply(_set_bn_eval)
    total, correct = 0, 0
    L_parts, Y_parts = [], []
    for x, y, m, pres, _ in loader:
        x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device).float()
        logits_y, logits_p = model(x, m)

        # --- accuracy ---
        correct += (logits_y.argmax(1) == y).sum().item()
        total   += y.numel()

        # --- presence F1 collection ---
        # strong guards (catch the kinds of bugs that produced 0.26 before)
        assert logits_p.ndim == 2, f"parts logits must be (B,C); got {logits_p.shape}"
        assert pres.ndim     == 2, f"presence must be (B,C); got {pres.shape}"
        assert logits_p.shape == pres.shape, f"shape mismatch {logits_p.shape} vs {pres.shape}"
        L_parts.append(logits_p.detach().cpu())
        Y_parts.append(pres.detach().cpu())

    acc = correct / max(total, 1)
    L = torch.cat(L_parts) if L_parts else torch.empty(0, device='cpu')
    Y = torch.cat(Y_parts) if Y_parts else torch.empty(0, device='cpu')
    f1 = _macro_f1_from_logits(L, Y, thr=thr) if L.numel() else float("nan")
    return acc, f1

@torch.no_grad()
def evaluate_baseline(model, loader, device, thr=0.5):
    model.eval(); model.apply(_set_bn_eval)
    total, correct = 0, 0
    L_parts, Y_parts = [], []
    for x, y, _m, pres, _ in loader:
        x, y, pres = x.to(device), y.to(device), pres.to(device).float()
        logits_y, logits_p = model(x)
        correct += (logits_y.argmax(1) == y).sum().item()
        total   += y.numel()
        assert logits_p.ndim == 2 and pres.ndim == 2 and logits_p.shape == pres.shape
        L_parts.append(logits_p.detach().cpu())
        Y_parts.append(pres.detach().cpu())
    acc = correct / max(total, 1)
    L = torch.cat(L_parts); Y = torch.cat(Y_parts)
    f1 = _macro_f1_from_logits(L, Y, thr=thr)
    return acc, f1

@torch.no_grad()
def evaluate_indep(model, loader, device, spec, super_list, thr=0.5):
    model.eval(); model.apply(_set_bn_eval)
    total, correct = 0, 0
    L_parts, Y_parts = [], []
    for x, y, m, pres, _ in loader:
        x, y, m, pres = x.to(device), y.to(device), m.to(device), pres.to(device).float()
        m_ind, y_ind = pack_shared_to_indep_batch(m, pres, y, spec, super_list, train_ds.part_types)
        logits_y, logits_i = model(x, m_ind)
        correct += (logits_y.argmax(1) == y).sum().item()
        total   += y.numel()
        logits_shared = indep_logits_to_shared(logits_i, y, spec, super_list)
        assert logits_shared.shape == pres.shape, f"indep→shared shape mismatch {logits_shared.shape} vs {pres.shape}"
        L_parts.append(logits_shared.detach().cpu())
        Y_parts.append(pres.detach().cpu())
    acc = correct / max(total, 1)
    L = torch.cat(L_parts); Y = torch.cat(Y_parts)
    f1 = _macro_f1_from_logits(L, Y, thr=thr)
    return acc, f1

In [129]:
class EarlyStopper:
    def __init__(self, patience=5, min_delta=0.0, mode="max", monitor="val_acc"):
        assert mode in {"max", "min"}
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.monitor = monitor
        self.best = -float("inf") if mode == "max" else float("inf")
        self.num_bad = 0
        self.stopped = False
        self.stopped_epoch = None

    def _is_improvement(self, value):
        if self.mode == "max":
            return value >= self.best + self.min_delta
        else:
            return value <= self.best - self.min_delta

    def step(self, metric_value, epoch):
        """Returns (should_stop, is_new_best)."""
        is_best = False
        if self._is_improvement(metric_value):
            self.best = metric_value
            self.num_bad = 0
            is_best = True
        else:
            self.num_bad += 1
            if self.num_bad >= self.patience:
                self.stopped = True
                self.stopped_epoch = epoch
        return self.stopped, is_best

In [130]:
def train_all_together(
    epochs=3,
    lr=1e-3,
    bs=32,
    alpha_part=0.1,         # weight for part presence BCE
    train_loader=None,
    val_loader=None,
    device=None
):
    dev = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build dataloaders if not provided (assumes train_ds / val_ds / collate_fn exist in the notebook)
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

    # Model setup
    num_labels = len(train_ds.supercats)
    num_shared_parts = len(PART_VOCAB)

    baseline = BaselineMTLResNet(num_labels, num_shared_parts).to(dev)
    shared   = SharedMTLResNet(num_labels, num_shared_parts).to(dev)

    # Independent spec based on supercats present in training set
    spec = build_indep_spec(train_ds.supercats)
    num_indep_parts = spec.K_max
    indep   = IndependentMTLResNet(num_labels, num_indep_parts).to(dev)

    opt_base   = torch.optim.Adam(baseline.parameters(), lr=lr)
    opt_shared = torch.optim.Adam(shared.parameters(),   lr=lr)
    opt_indep  = torch.optim.Adam(indep.parameters(),    lr=lr)

    # Logging
    rows = []

    for ep in range(1, epochs + 1):
        baseline.train(); shared.train(); indep.train()

        # Running stats (train)
        tr_acc_base, tr_f1_base = [], []
        tr_acc_shared, tr_f1_shared = [], []
        tr_acc_indep, tr_f1_indep = [], []

        for step, (x, y, m_shared, pres_shared, _haspix) in enumerate(tl, start=1):
            x, y = x.to(dev), y.to(dev)
            m_shared, pres_shared = m_shared.to(dev), pres_shared.to(dev)

            # ----- Baseline (RGB only) -----
            logits_y_b, logits_p_b = baseline(x)
            loss_b = F.cross_entropy(logits_y_b, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_p_b, pres_shared)
            opt_base.zero_grad()
            loss_b.backward()
            opt_base.step()

            tr_acc_base.append(acc_from_logits(logits_y_b, y))
            tr_f1_base.append(macro_presence_f1_from_logits_shared(logits_p_b, pres_shared))

            # ----- Shared-channels (RGB + shared masks) -----
            logits_y_s, logits_p_s = shared(x, m_shared)
            loss_s = F.cross_entropy(logits_y_s, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_p_s, pres_shared)
            opt_shared.zero_grad()
            loss_s.backward()
            opt_shared.step()

            tr_acc_shared.append(acc_from_logits(logits_y_s, y))
            tr_f1_shared.append(macro_presence_f1_from_logits_shared(logits_p_s, pres_shared))

            # ----- Independent-channels (RGB + (super, part) channels) -----
            # pack the current batch’s shared masks/presence -> independent layout for each sample’s supercategory
            m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, train_ds.supercats, train_ds.part_types)
            logits_y_i, logits_i = indep(x, m_ind)
            loss_i = F.cross_entropy(logits_y_i, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_i, y_ind)
            opt_indep.zero_grad()
            loss_i.backward()
            opt_indep.step()

            # For F1 we compare in SHARED space for apples-to-apples
            logits_shared_i = indep_logits_to_shared(logits_i, y, spec, train_ds.supercats)
            tr_acc_indep.append(acc_from_logits(logits_y_i, y))
            tr_f1_indep.append(macro_presence_f1_from_logits_shared(logits_shared_i, pres_shared))

            # (Optional) occasional progress print
            if step % 100 == 0:
                print(f"Epoch {ep:02d} | Step {step:04d} "
                      f"| base acc {np.mean(tr_acc_base):.3f} shared acc {np.mean(tr_acc_shared):.3f} indep acc {np.mean(tr_acc_indep):.3f}")

        # ---- End of epoch: evaluate on val ----
        va_acc_base, va_f1_base     = evaluate_baseline(baseline, vl, dev)
        va_acc_shared, va_f1_shared = evaluate_shared(shared, vl, dev)
        va_acc_indep, va_f1_indep   = evaluate_indep(indep, vl, dev, spec, val_ds.supercats)

        # Aggregate train stats (averages across batches)
        tr_acc_base_m,   tr_f1_base_m   = float(np.mean(tr_acc_base)),   float(np.mean(tr_f1_base))
        tr_acc_shared_m, tr_f1_shared_m = float(np.mean(tr_acc_shared)), float(np.mean(tr_f1_shared))
        tr_acc_indep_m,  tr_f1_indep_m  = float(np.mean(tr_acc_indep)),  float(np.mean(tr_f1_indep))

        print(f"[E{ep:02d}] BASE:   train acc {tr_acc_base_m:6.2%} | val acc {va_acc_base:6.2%} | train F1 {tr_f1_base_m:.3f} | val F1 {va_f1_base:.3f}")
        print(f"[E{ep:02d}] SHARED: train acc {tr_acc_shared_m:6.2%} | val acc {va_acc_shared:6.2%} | train F1 {tr_f1_shared_m:.3f} | val F1 {va_f1_shared:.3f}")
        print(f"[E{ep:02d}] INDEP:  train acc {tr_acc_indep_m:6.2%} | val acc {va_acc_indep:6.2%} | train F1 {tr_f1_indep_m:.3f} | val F1 {va_f1_indep:.3f}")

        rows.append({
            "epoch": ep,

            "BASE_train_acc":   tr_acc_base_m,
            "BASE_val_acc":     va_acc_base,
            "BASE_train_f1":    tr_f1_base_m,
            "BASE_val_f1":      va_f1_base,

            "SHARED_train_acc": tr_acc_shared_m,
            "SHARED_val_acc":   va_acc_shared,
            "SHARED_train_f1":  tr_f1_shared_m,
            "SHARED_val_f1":    va_f1_shared,

            "INDEP_train_acc":  tr_acc_indep_m,
            "INDEP_val_acc":    va_acc_indep,
            "INDEP_train_f1":   tr_f1_indep_m,
            "INDEP_val_f1":     va_f1_indep,
        })

    results = pd.DataFrame(rows)
    display(results)

    # Quick summary of best validation performance per model
    best_summary = pd.DataFrame([
        {
            "Model": "Baseline (RGB)",
            "Best Val Acc": results["BASE_val_acc"].max(),
            "Best Val F1":  results["BASE_val_f1"].max(),
        },
        {
            "Model": "Shared-channels",
            "Best Val Acc": results["SHARED_val_acc"].max(),
            "Best Val F1":  results["SHARED_val_f1"].max(),
        },
        {
            "Model": "Independent-channels",
            "Best Val Acc": results["INDEP_val_acc"].max(),
            "Best Val F1":  results["INDEP_val_f1"].max(),
        },
    ])
    display(best_summary)

    return {
        "history": results,
        "best": best_summary,
        "models": {"baseline": baseline, "shared": shared, "indep": indep},
        "spec": spec,
    }

In [131]:
ROOT = "PartImageNet_Seg/PartImageNet"

TRAIN_JSON     = os.path.join(ROOT, "annotations/train/train.json")
VAL_JSON       = os.path.join(ROOT, "annotations/val/val.json")
TRAIN_IMG_ROOT = os.path.join(ROOT, "images/train")
VAL_IMG_ROOT   = os.path.join(ROOT, "images/val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = PartImageNetCOCODataset(
    ann_path=TRAIN_JSON,
    img_root=TRAIN_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=True,
    shared_by_type=True,
)
val_ds = PartImageNetCOCODataset(
    ann_path=VAL_JSON,
    img_root=VAL_IMG_ROOT,
    img_size=IMG_SIZE,
    augment=False,
    shared_by_type=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          collate_fn=collate_fn, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                          collate_fn=collate_fn, drop_last=False)

print("\nRunning audit on training data (first 200 images)…")
audit_masks(train_ds, max_images=200)

[init] ann_path=PartImageNet_Seg/PartImageNet/annotations/train/train.json
[init] images=20481 | categories=40
[init] Recognized part types (shared): ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
[init] Shared-by-type part channels: 13; Total input channels: 16
[init] ann_path=PartImageNet_Seg/PartImageNet/annotations/val/val.json
[init] images=1206 | categories=40
[init] Recognized part types (shared): ['body', 'engine', 'fin', 'foot', 'hand', 'head', 'mouth', 'sail', 'seat', 'side_mirror', 'tail', 'tire', 'wing']
[init] Shared-by-type part channels: 13; Total input channels: 16

Running audit on training data (first 200 images)…
[audit] images scanned: 200
[audit] images with any foreground mask: 200
[audit] approx. part-channels with any pixels across scanned images: 3


In [132]:
def train_all_together(
    epochs=3,
    lr=1e-3,
    bs=32,
    alpha_part=0.1,         # weight for part presence BCE
    train_loader=None,
    val_loader=None,
    device=None,
    save_dir: str = "checkpoints",
    ckpt_prefix: str = "partimagenet",
    save_best_by: str = "acc",   # 'acc' or 'f1'
    save_last: bool = True
):
    dev = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build dataloaders if not provided (assumes train_ds / val_ds / collate_fn exist in the notebook)
    tl = train_loader or DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn)
    vl = val_loader   or DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

    # Model setup
    num_labels = len(train_ds.supercats)
    num_shared_parts = len(PART_VOCAB)

    baseline = BaselineMTLResNet(num_labels, num_shared_parts).to(dev)
    shared   = SharedMTLResNet(num_labels, num_shared_parts).to(dev)

    # Independent spec based on supercats present in training set
    spec = build_indep_spec(train_ds.supercats)
    num_indep_parts = spec.K_max
    indep   = IndependentMTLResNet(num_labels, num_indep_parts).to(dev)

    opt_base   = torch.optim.Adam(baseline.parameters(), lr=lr)
    opt_shared = torch.optim.Adam(shared.parameters(),   lr=lr)
    opt_indep  = torch.optim.Adam(indep.parameters(),    lr=lr)

    # Logging
    rows = []
    # --- Checkpointing setup ---
    os.makedirs(save_dir, exist_ok=True)
    best_by = save_best_by.lower()
    assert best_by in ("acc","f1"), "save_best_by must be 'acc' or 'f1'"
    best_BASE = -1.0; best_SHARED = -1.0; best_INDEP = -1.0
    best_ep_BASE = 0; best_ep_SHARED = 0; best_ep_INDEP = 0

    def _save_ckpt(model, path, epoch, metrics: dict):
        meta = {
            "state_dict": model.state_dict(),
            "epoch": epoch,
            "metrics": metrics,
            "supercats": getattr(train_ds, "supercats", None),
            "part_vocab": PART_VOCAB if 'PART_VOCAB' in globals() else None,
        }
        torch.save(meta, path)


    for ep in range(1, epochs + 1):
        baseline.train(); shared.train(); indep.train()

        # Running stats (train)
        tr_acc_base, tr_f1_base = [], []
        tr_acc_shared, tr_f1_shared = [], []
        tr_acc_indep, tr_f1_indep = [], []

        for step, (x, y, m_shared, pres_shared, _haspix) in enumerate(tl, start=1):
            x, y = x.to(dev), y.to(dev)
            m_shared, pres_shared = m_shared.to(dev), pres_shared.to(dev)

            # ----- Baseline (RGB only) -----
            logits_y_b, logits_p_b = baseline(x)
            loss_b = F.cross_entropy(logits_y_b, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_p_b, pres_shared)
            opt_base.zero_grad()
            loss_b.backward()
            opt_base.step()

            tr_acc_base.append(acc_from_logits(logits_y_b, y))
            tr_f1_base.append(macro_presence_f1_from_logits_shared(logits_p_b, pres_shared))

            # ----- Shared-channels (RGB + shared masks) -----
            logits_y_s, logits_p_s = shared(x, m_shared)
            loss_s = F.cross_entropy(logits_y_s, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_p_s, pres_shared)
            opt_shared.zero_grad()
            loss_s.backward()
            opt_shared.step()

            tr_acc_shared.append(acc_from_logits(logits_y_s, y))
            tr_f1_shared.append(macro_presence_f1_from_logits_shared(logits_p_s, pres_shared))

            # ----- Independent-channels (RGB + (super, part) channels) -----
            # pack the current batch’s shared masks/presence -> independent layout for each sample’s supercategory
            m_ind, y_ind = pack_shared_to_indep_batch(m_shared, pres_shared, y, spec, train_ds.supercats, train_ds.part_types)
            logits_y_i, logits_i = indep(x, m_ind)
            loss_i = F.cross_entropy(logits_y_i, y) + alpha_part * F.binary_cross_entropy_with_logits(logits_i, y_ind)
            opt_indep.zero_grad()
            loss_i.backward()
            opt_indep.step()

            # For F1 we compare in SHARED space for apples-to-apples
            logits_shared_i = indep_logits_to_shared(logits_i, y, spec, train_ds.supercats)
            tr_acc_indep.append(acc_from_logits(logits_y_i, y))
            tr_f1_indep.append(macro_presence_f1_from_logits_shared(logits_shared_i, pres_shared))

            # (Optional) occasional progress print
            if step % 100 == 0:
                print(f"Epoch {ep:02d} | Step {step:04d} "
                      f"| base acc {np.mean(tr_acc_base):.3f} shared acc {np.mean(tr_acc_shared):.3f} indep acc {np.mean(tr_acc_indep):.3f}")

        # ---- End of epoch: evaluate on val ----
        va_acc_base, va_f1_base     = evaluate_baseline(baseline, vl, dev)
        va_acc_shared, va_f1_shared = evaluate_shared(shared, vl, dev)
        va_acc_indep, va_f1_indep   = evaluate_indep(indep, vl, dev, spec, val_ds.supercats)
        # ---- Save checkpoints ----
        metric_BASE  = va_acc_base  if best_by == "acc" else va_f1_base
        metric_SHARED= va_acc_shared if best_by == "acc" else va_f1_shared
        metric_INDEP = va_acc_indep if best_by == "acc" else va_f1_indep

        # Save 'last' snapshots for each model
        if save_last:
            _save_ckpt(baseline, os.path.join(save_dir, f"{ckpt_prefix}_BASE_last.ckpt"),   ep,
                       {"val_acc": va_acc_base, "val_f1": va_f1_base})
            _save_ckpt(shared,   os.path.join(save_dir, f"{ckpt_prefix}_SHARED_last.ckpt"), ep,
                       {"val_acc": va_acc_shared, "val_f1": va_f1_shared})
            _save_ckpt(indep,    os.path.join(save_dir, f"{ckpt_prefix}_INDEP_last.ckpt"),  ep,
                       {"val_acc": va_acc_indep, "val_f1": va_f1_indep, "indep_num_parts": spec.K_max})

        # Save 'best' by chosen metric
        if metric_BASE > best_BASE:
            best_BASE, best_ep_BASE = metric_BASE, ep
            _save_ckpt(baseline, os.path.join(save_dir, f"{ckpt_prefix}_BASE_best.ckpt"), ep,
                       {"val_acc": va_acc_base, "val_f1": va_f1_base, "best_by": best_by})
            print(f"  ↳ [BASE] new best {best_by}={metric_BASE:.4f} @epoch {ep}")
        if metric_SHARED > best_SHARED:
            best_SHARED, best_ep_SHARED = metric_SHARED, ep
            _save_ckpt(shared, os.path.join(save_dir, f"{ckpt_prefix}_SHARED_best.ckpt"), ep,
                       {"val_acc": va_acc_shared, "val_f1": va_f1_shared, "best_by": best_by})
            print(f"  ↳ [SHARED] new best {best_by}={metric_SHARED:.4f} @epoch {ep}")
        if metric_INDEP > best_INDEP:
            best_INDEP, best_ep_INDEP = metric_INDEP, ep
            _save_ckpt(indep, os.path.join(save_dir, f"{ckpt_prefix}_INDEP_best.ckpt"), ep,
                       {"val_acc": va_acc_indep, "val_f1": va_f1_indep, "best_by": best_by, "indep_num_parts": spec.K_max})
            print(f"  ↳ [INDEP] new best {best_by}={metric_INDEP:.4f} @epoch {ep}")


        # Aggregate train stats (averages across batches)
        tr_acc_base_m,   tr_f1_base_m   = float(np.mean(tr_acc_base)),   float(np.mean(tr_f1_base))
        tr_acc_shared_m, tr_f1_shared_m = float(np.mean(tr_acc_shared)), float(np.mean(tr_f1_shared))
        tr_acc_indep_m,  tr_f1_indep_m  = float(np.mean(tr_acc_indep)),  float(np.mean(tr_f1_indep))

        print(f"[E{ep:02d}] BASE:   train acc {tr_acc_base_m:6.2%} | val acc {va_acc_base:6.2%} | train F1 {tr_f1_base_m:.3f} | val F1 {va_f1_base:.3f}")
        print(f"[E{ep:02d}] SHARED: train acc {tr_acc_shared_m:6.2%} | val acc {va_acc_shared:6.2%} | train F1 {tr_f1_shared_m:.3f} | val F1 {va_f1_shared:.3f}")
        print(f"[E{ep:02d}] INDEP:  train acc {tr_acc_indep_m:6.2%} | val acc {va_acc_indep:6.2%} | train F1 {tr_f1_indep_m:.3f} | val F1 {va_f1_indep:.3f}")

        rows.append({
            "epoch": ep,

            "BASE_train_acc":   tr_acc_base_m,
            "BASE_val_acc":     va_acc_base,
            "BASE_train_f1":    tr_f1_base_m,
            "BASE_val_f1":      va_f1_base,

            "SHARED_train_acc": tr_acc_shared_m,
            "SHARED_val_acc":   va_acc_shared,
            "SHARED_train_f1":  tr_f1_shared_m,
            "SHARED_val_f1":    va_f1_shared,

            "INDEP_train_acc":  tr_acc_indep_m,
            "INDEP_val_acc":    va_acc_indep,
            "INDEP_train_f1":   tr_f1_indep_m,
            "INDEP_val_f1":     va_f1_indep,
        })

    results = pd.DataFrame(rows)
    display(results)

    # Quick summary of best validation performance per model
    best_summary = pd.DataFrame([
        {
            "Model": "Baseline (RGB)",
            "Best Val Acc": results["BASE_val_acc"].max(),
            "Best Val F1":  results["BASE_val_f1"].max(),
        },
        {
            "Model": "Shared-channels",
            "Best Val Acc": results["SHARED_val_acc"].max(),
            "Best Val F1":  results["SHARED_val_f1"].max(),
        },
        {
            "Model": "Independent-channels",
            "Best Val Acc": results["INDEP_val_acc"].max(),
            "Best Val F1":  results["INDEP_val_f1"].max(),
        },
    ])
    display(best_summary)

    return {
        "history": results,
        "best": best_summary,
        "models": {"baseline": baseline, "shared": shared, "indep": indep},
        "spec": spec,
    }

In [133]:
result = train_all_together(
    epochs=8,
    lr=3e-4,
    bs=32,
    alpha_part=0.1,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    save_dir="checkpoints",
    ckpt_prefix="partimagenet_run1",
    save_best_by="acc",   # or "f1"
    save_last=True,
)

Epoch 01 | Step 0100 | base acc 0.703 shared acc 0.290 indep acc 0.295
Epoch 01 | Step 0200 | base acc 0.733 shared acc 0.331 indep acc 0.326
Epoch 01 | Step 0300 | base acc 0.746 shared acc 0.346 indep acc 0.345
Epoch 01 | Step 0400 | base acc 0.757 shared acc 0.364 indep acc 0.358
Epoch 01 | Step 0500 | base acc 0.764 shared acc 0.384 indep acc 0.375
Epoch 01 | Step 0600 | base acc 0.767 shared acc 0.397 indep acc 0.390
Epoch 01 | Step 0700 | base acc 0.772 shared acc 0.408 indep acc 0.402
Epoch 01 | Step 0800 | base acc 0.777 shared acc 0.420 indep acc 0.412
Epoch 01 | Step 0900 | base acc 0.781 shared acc 0.433 indep acc 0.420
Epoch 01 | Step 1000 | base acc 0.783 shared acc 0.443 indep acc 0.429
Epoch 01 | Step 1100 | base acc 0.787 shared acc 0.455 indep acc 0.442
Epoch 01 | Step 1200 | base acc 0.788 shared acc 0.468 indep acc 0.452
  ↳ [BASE] new best acc=0.8367 @epoch 1
  ↳ [SHARED] new best acc=0.4262 @epoch 1
  ↳ [INDEP] new best acc=0.6169 @epoch 1
[E01] BASE:   train acc 7

,epoch,BASE_train_acc,BASE_val_acc,BASE_train_f1,BASE_val_f1,SHARED_train_acc,SHARED_val_acc,SHARED_train_f1,SHARED_val_f1,INDEP_train_acc,INDEP_val_acc,INDEP_train_f1,INDEP_val_f1
0,1,0.787251,0.836650,0.290231,0.297249,0.479722,0.426202,0.281115,0.447192,0.462298,0.616915,0.117843,0.213691
1,2,0.842106,0.812604,0.290229,0.297249,0.740502,0.592869,0.451177,0.680614,0.678451,0.407960,0.205381,0.237131
2,3,0.865229,0.841625,0.290338,0.297249,0.814120,0.805970,0.525631,0.795885,0.735288,0.535655,0.218418,0.277789
3,4,0.876122,0.859038,0.290213,0.297249,0.838320,0.729685,0.544338,0.752387,0.755588,0.781095,0.221280,0.354521
4,5,0.892841,0.844942,0.290263,0.297249,0.849346,0.810945,0.553389,0.781529,0.777794,0.729685,0.227395,0.355011
5,6,0.903555,0.868159,0.290323,0.297249,0.869627,0.856551,0.564272,0.813839,0.792350,0.821725,0.230215,0.353502
6,7,0.911436,0.861526,0.290279,0.297249,0.870305,0.693201,0.561439,0.736770,0.806744,0.624378,0.229075,0.319939
7,8,0.920576,0.876451,0.290138,0.297249,0.879749,0.849088,0.566911,0.755886,0.815928,0.802653,0.230279,0.377768


,Model,Best Val Acc,Best Val F1
0,Baseline (RGB),0.876451,0.297249
1,Shared-channels,0.856551,0.813839
2,Independent-channels,0.821725,0.377768


In [134]:
out = train_all_together(epochs=10, lr=1e-3, bs=32, train_loader=train_loader, val_loader=val_loader, device=device)

Epoch 01 | Step 0100 | base acc 0.390 shared acc 0.269 indep acc 0.242
Epoch 01 | Step 0200 | base acc 0.456 shared acc 0.300 indep acc 0.288
Epoch 01 | Step 0300 | base acc 0.482 shared acc 0.317 indep acc 0.308
Epoch 01 | Step 0400 | base acc 0.500 shared acc 0.329 indep acc 0.320
Epoch 01 | Step 0500 | base acc 0.509 shared acc 0.340 indep acc 0.331
Epoch 01 | Step 0600 | base acc 0.524 shared acc 0.349 indep acc 0.335
Epoch 01 | Step 0700 | base acc 0.532 shared acc 0.359 indep acc 0.345
Epoch 01 | Step 0800 | base acc 0.541 shared acc 0.365 indep acc 0.354
Epoch 01 | Step 0900 | base acc 0.550 shared acc 0.371 indep acc 0.363
Epoch 01 | Step 1000 | base acc 0.556 shared acc 0.377 indep acc 0.374
Epoch 01 | Step 1100 | base acc 0.563 shared acc 0.386 indep acc 0.385
Epoch 01 | Step 1200 | base acc 0.571 shared acc 0.394 indep acc 0.398
  ↳ [BASE] new best acc=0.5813 @epoch 1
  ↳ [SHARED] new best acc=0.2164 @epoch 1
  ↳ [INDEP] new best acc=0.1592 @epoch 1
[E01] BASE:   train acc 5

,epoch,BASE_train_acc,BASE_val_acc,BASE_train_f1,BASE_val_f1,SHARED_train_acc,SHARED_val_acc,SHARED_train_f1,SHARED_val_f1,INDEP_train_acc,INDEP_val_acc,INDEP_train_f1,INDEP_val_f1
0,1,0.574756,0.581260,0.290271,0.297249,0.400433,0.216418,0.266647,0.324387,0.407397,0.159204,0.117419,0.143697
1,2,0.692395,0.647595,0.290243,0.297249,0.642350,0.402985,0.422755,0.453838,0.631980,0.671642,0.204883,0.316569
2,3,0.731216,0.733002,0.290385,0.297249,0.778776,0.739635,0.517228,0.734257,0.699623,0.631012,0.219618,0.287385
3,4,0.765538,0.773632,0.290257,0.297249,0.816348,0.769486,0.539152,0.758830,0.733291,0.339967,0.224512,0.213860
4,5,0.785721,0.730514,0.290227,0.297249,0.833135,0.810116,0.547274,0.827032,0.751792,0.679104,0.228490,0.356183
5,6,0.806652,0.752073,0.290067,0.297249,0.848741,0.631841,0.560187,0.562788,0.769714,0.468491,0.230505,0.276578
6,7,0.820898,0.751244,0.290223,0.297249,0.856245,0.759536,0.563356,0.701788,0.780455,0.784411,0.228963,0.358234
7,8,0.837965,0.781095,0.290313,0.297249,0.870641,0.816750,0.573768,0.825784,0.796240,0.785240,0.233512,0.363468
8,9,0.855617,0.776119,0.290190,0.297249,0.877316,0.766998,0.571112,0.715557,0.806437,0.772803,0.233591,0.362645
9,10,0.867095,0.781095,0.290362,0.297249,0.884501,0.471808,0.578326,0.475418,0.815222,0.208955,0.235950,0.136073


,Model,Best Val Acc,Best Val F1
0,Baseline (RGB),0.781095,0.297249
1,Shared-channels,0.816750,0.827032
2,Independent-channels,0.785240,0.363468


## Integrating Self-Annotation

## Cue-Conflict

In [32]:
import os, csv, hashlib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models as tvm
from PIL import Image


ROOT = "model-vs-human/datasets/cue-conflict"   # contains airplane/, bear/, bicycle/, ...
CKPT_BASE_PATH   = "checkpoints/partimagenet_run1_BASE_best.ckpt"
CKPT_SHARED_PATH = "checkpoints/partimagenet_run1_SHARED_best.ckpt"
CKPT_INDEP_PATH  = "checkpoints/partimagenet_run1_INDEP_best.ckpt"

BATCH_SIZE  = 32
NUM_WORKERS = 0          # keep 0 until caches fill (WSL-safe)
IMG_SIZE    = 224
USE_SOFT_MASKS = True

# Small DINOv3 (ViT-S/16+) URL you provided (lighter/faster)
CKPT_URL_VITS16 = (
    "https://dinov3.llamameta.net/dinov3_vits16plus/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth?"
    "Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiM3pieWFia3h5eGwzNG05a214MTk5MGdnIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTU3MzQ0MTZ9fX1dfQ__&Signature=hjTucjqUXJCdgPVKHS7STgmqFouOWJpy2NLMZlVX83k5HSblY6ljppV7z2ayPqrXNl5L5qb%7ESFwIlEbKt6URx9%7EryajYP-ERnamT8toBMH5yQ0v37g8IFEwoHkE9QwfBDuFbvY0iR%7EDkjw4x1DSN95OTPwxPQGZfKiDdjdYdDg5KoZVwIpEwCWrHZXkULsigkXHcJror2fEC9wXYYTp6SanBtx%7EExh3KhAmABsZ%7Eo3bjaqU82ceLXf5i5frzGrwqXQbOGJOtMbuybSaEYVY3iz2QExJtywy7dQ2WcmUnQ%7EcLHTRfJRMEeVJB2CtIY4vVovOxT59Hjew9dkDcTRVffQ__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1271352317331045"
)

In [33]:
def _interp_pos(ckpt_pos: torch.Tensor, model_pos: torch.Tensor) -> torch.Tensor:
    if ckpt_pos.ndim!=3 or model_pos.ndim!=3: return model_pos
    def split_cls(pe): return pe[:, :1], pe[:, 1:]
    cls_m, grid_m = split_cls(model_pos); _, grid_c = split_cls(ckpt_pos)
    s = int((grid_c.shape[1])**0.5); S = int((grid_m.shape[1])**0.5)
    grid_c = grid_c.transpose(1,2).reshape(1, grid_c.shape[2], s, s)
    grid_i = F.interpolate(grid_c, size=(S,S), mode="bicubic", align_corners=False)
    grid_i = grid_i.reshape(1, grid_i.shape[1], S*S).transpose(1,2)
    return torch.cat([cls_m, grid_i], dim=1)

def build_vits16_from_url(url: str, device=torch.device("cpu"), dtype=torch.float32):
    import timm, torch
    vit = timm.create_model("vit_small_patch16_224", pretrained=False)
    vit.reset_classifier(0)
    vit.eval().to(device=device, dtype=dtype)
    sd = torch.hub.load_state_dict_from_url(url, map_location="cpu", check_hash=False)
    if isinstance(sd, dict):
        for k in ("state_dict","model","teacher","student","backbone"):
            if k in sd and isinstance(sd[k], dict): sd = sd[k]; break
    cleaned = {}
    for k,v in sd.items():
        for p in ("module.","backbone.","encoder.","student_backbone."):
            if k.startswith(p): k = k[len(p):]
        cleaned[k] = v
    if "patch_embed.proj.weight" in cleaned and hasattr(vit,"patch_embed"):
        if cleaned["patch_embed.proj.weight"].shape != vit.patch_embed.proj.weight.shape:
            cleaned.pop("patch_embed.proj.weight", None); cleaned.pop("patch_embed.proj.bias", None)
    vit.load_state_dict(cleaned, strict=False)
    if "pos_embed" in cleaned and cleaned["pos_embed"].shape != vit.pos_embed.shape:
        with torch.no_grad(): vit.pos_embed.copy_(_interp_pos(cleaned["pos_embed"], vit.pos_embed))
        vit.load_state_dict(cleaned, strict=False)
    return vit

@torch.no_grad()
def vit_patch_tokens(vit: torch.nn.Module, images: torch.Tensor):
    """
    Returns:
      tokens: (B, N, D) patch tokens (CLS removed)
      hw:     (Ht, Wt) patch grid size
    Works with timm ViT where PatchEmbed may return (B,N,D) OR (B,D,Ht,Wt).
    """
    x = vit.patch_embed(images)

    if x.ndim == 4:
        # timm PatchEmbed(flatten=False) → (B, D, Ht, Wt)
        B, D, Ht, Wt = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, N, D)
    elif x.ndim == 3:
        # timm PatchEmbed(flatten=True) → (B, N, D)
        B, N, D = x.shape
        # Try to read grid size from module; otherwise infer from N
        grid = getattr(vit.patch_embed, "grid_size", None)
        if isinstance(grid, (tuple, list)) and len(grid) == 2:
            Ht, Wt = int(grid[0]), int(grid[1])
        else:
            # Fallback: assume square grid
            Ht = Wt = int(N ** 0.5)
    else:
        raise ValueError(f"Unexpected patch_embed output shape: {tuple(x.shape)}")

    # Add CLS + position if present (timm ViT)
    if getattr(vit, "cls_token", None) is not None:
        x = torch.cat([vit.cls_token.expand(B, -1, -1), x], dim=1)  # (B, 1+N, D)
    if getattr(vit, "pos_embed", None) is not None:
        pe = vit.pos_embed[:, :x.size(1), :].to(dtype=x.dtype)
        x = x + pe
    if getattr(vit, "pos_drop", None) is not None:
        x = vit.pos_drop(x)

    for blk in vit.blocks:
        x = blk(x)

    if getattr(vit, "norm", None) is not None:
        x = vit.norm(x)

    # remove CLS if still present
    if x.size(1) == (Ht * Wt + 1):
        x = x[:, 1:, :]

    return x, (Ht, Wt)

@torch.no_grad()
def kmeans_one(x, k: int, iters: int=12, generator=None):
    N,D = x.shape
    if generator is None:
        generator = torch.Generator(device=x.device); generator.manual_seed(0)
    idx = torch.randperm(N, generator=generator, device=x.device)[:k]
    c = F.normalize(x[idx], dim=-1)
    for _ in range(iters):
        labels = (x @ c.T).argmax(dim=1)
        new_c = torch.stack([ x[labels==i].mean(0) if (labels==i).any() else c[i] for i in range(k) ])
        c = F.normalize(new_c, dim=-1)
    return labels, c

def soft_assign(x, c, T=0.07):
    x = F.normalize(x, dim=-1); c = F.normalize(c, dim=-1)
    return ((x @ c.T)/T).softmax(dim=-1)

def soft_to_masks(prob, hw):
    N,K = prob.shape; Ht,Wt = hw
    return prob.transpose(0,1).reshape(K,Ht,Wt)

def upsample(m, out_hw):
    return F.interpolate(m, size=out_hw, mode="bilinear", align_corners=False)

class DinoSelfAnnotator(torch.nn.Module):
    def __init__(self, vit, num_parts=6, kmeans_iters=12, soft_temp=0.07):
        super().__init__(); self.vit=vit; self.num_parts=num_parts; self.kmeans_iters=kmeans_iters; self.soft_temp=soft_temp
    @torch.no_grad()
    def forward(self, images):
        B,_,H,W = images.shape
        tokens, (Ht,Wt) = vit_patch_tokens(self.vit, images)
        tokens_n = F.normalize(tokens, dim=-1)
        softL=[]
        for b in range(B):
            lbl,c = kmeans_one(tokens_n[b], k=self.num_parts, iters=self.kmeans_iters)
            soft = soft_to_masks(soft_assign(tokens_n[b], c, T=self.soft_temp),(Ht,Wt))
            softL.append(upsample(soft[None], (H,W))[0])
        return {"soft_masks": torch.stack(softL,0)}   # (B,P,H,W)

In [34]:
def _load_raw(path):
    raw = torch.load(path, map_location="cpu")
    return raw.get("state_dict", raw), raw   # (state_dict, meta)

def _infer_num_labels(sd, meta):
    if isinstance(meta, dict) and meta.get("supercats") is not None:
        return len(meta["supercats"]), meta["supercats"]
    # fallback from head weight
    for k in ("fc_label.weight","head.weight","classifier.weight","fc.weight"):
        if k in sd: 
            return sd[k].shape[0], None
    # last linear
    nl = None
    for k,v in sd.items():
        if k.endswith("weight") and v.ndim==2: nl = v.shape[0]
    return nl, None

def _first_conv_in_ch(sd):
    # find the backbone's first conv weight (4D)
    for k,v in sd.items():
        if k.endswith("conv1.weight") and v.ndim==4:
            return v.shape[1]
    raise RuntimeError("conv1.weight not found")

def load_shared_ckpt(path, device):
    sd, meta = _load_raw(path)
    in_ch = _first_conv_in_ch(sd)     # 3 + K_shared
    K = in_ch - 3
    num_labels, supercats = _infer_num_labels(sd, meta)
    # arg names per your def:
    model = SharedMTLResNet(num_labels=num_labels, num_parts=K).to(device)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:    print("[shared] missing:", [m for m in missing if "num_batches_tracked" not in m][:8], "...")
    if unexpected: print("[shared] unexpected:", unexpected[:8], "...")
    model.eval()
    return model, K, num_labels, (supercats or [])

def load_baseline_ckpt(path, device, num_parts, num_labels_hint=None):
    sd, meta = _load_raw(path)
    if num_labels_hint is None:
        num_labels, _ = _infer_num_labels(sd, meta)
    else:
        num_labels = num_labels_hint
    model = BaselineMTLResNet(num_labels=num_labels, num_parts=int(num_parts)).to(device)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:    print("[base] missing:", [m for m in missing if "num_batches_tracked" not in m][:8], "...")
    if unexpected: print("[base] unexpected:", unexpected[:8], "...")
    model.eval()
    return model, int(num_parts), num_labels

def load_indep_ckpt(path, device):
    sd, meta = _load_raw(path)
    in_ch = _first_conv_in_ch(sd)     # 3 + K_indep
    K = in_ch - 3
    num_labels, _ = _infer_num_labels(sd, meta)
    model = IndependentMTLResNet(num_labels=num_labels, num_indep_parts=int(K)).to(device)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:    print("[indep] missing:", [m for m in missing if "num_batches_tracked" not in m][:8], "...")
    if unexpected: print("[indep] unexpected:", unexpected[:8], "...")
    model.eval()
    return model, int(K), num_labels


In [35]:
def _hash_key(path: str, K: int, size: int):
    return hashlib.md5(f"{path}|{K}|{size}".encode()).hexdigest()[:16]

class SharedMaskDataset(Dataset):
    """Returns (x_rgb, m_shared, y, path) where:
       x_rgb:    (3,H,W)
       m_shared: (P=K_shared,H,W) from DINOv3 (cached)
    """
    def __init__(self, root, annotator, K_shared, out_size=224, cache_dir="./mask_cache_shared"):
        self.base = datasets.ImageFolder(root)
        self.annot = annotator; self.P = int(K_shared)
        self.out = out_size
        self.cache = cache_dir
        os.makedirs(self.cache, exist_ok=True)
        self.rgb_tf = transforms.Compose([
            transforms.Resize(out_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(out_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ])
        self.vit_tf = transforms.Compose([
            transforms.Resize(out_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(out_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
        ])
        self.classes = self.base.classes

    def __len__(self): return len(self.base)

    def _cache_path(self, path): 
        return os.path.join(self.cache, f"{_hash_key(path, self.P, self.out)}.pt")

    def __getitem__(self, i):
        img, y = self.base[i]; path,_ = self.base.samples[i]
        if not isinstance(img, Image.Image): img = transforms.ToPILImage()(img)
        x_rgb = self.rgb_tf(img)
        f = self._cache_path(path)
        if os.path.isfile(f):
            m = torch.load(f, map_location="cpu")
        else:
            with torch.inference_mode():
                vi = self.vit_tf(img).unsqueeze(0)
                out = self.annot(vi)
                m = out["soft_masks"][0].clamp(0,1).cpu()   # (P,H,W)
            torch.save(m, f)
        return x_rgb, m, y, path
    
import re
def _norm(s: str) -> str:
    # normalize names to be robust to hyphens/underscores/case
    return re.sub(r"[^a-z0-9]+", "", s.lower())

def shared_to_indep_masks(m_shared: torch.Tensor,
                          y: torch.Tensor,
                          spec,
                          ds_classes: list[str]) -> torch.Tensor:
    """
    m_shared: (B, P, H, W)  shared part masks
    y:        (B,)          ImageFolder class indices (into ds_classes)
    spec:     IndepSpec from build_indep_spec(super_list) with:
                - spec.tuples: list[(super_name, part_name)]
                - spec.part_to_shared[part_name] -> shared part index
    ds_classes: list of class names from your dataset (ImageFolder)

    returns m_ind: (B, K, H, W) aligned to spec.tuples
    """
    B, P, H, W = m_shared.shape
    K = spec.K_max
    m_ind = torch.zeros(B, K, H, W, device=m_shared.device, dtype=m_shared.dtype)

    # group spec tuples by normalized superclass name, map to (t_idx, shared_part_idx)
    by_super = {}
    for t_idx, (sup_name, part_name) in enumerate(spec.tuples):
        ns = _norm(sup_name)
        p_idx = spec.part_to_shared[part_name]  # shared index for this part
        by_super.setdefault(ns, []).append((t_idx, p_idx))

    for b in range(B):
        cls_name = ds_classes[int(y[b].item())]
        ns = _norm(cls_name)
        pairs = by_super.get(ns, [])
        # If a class name isn’t in the spec, we just leave zeros (safe fallback)
        for t_idx, p_idx in pairs:
            if 0 <= p_idx < P:
                m_ind[b, t_idx] = m_shared[b, p_idx]
    return m_ind

## On Cue-Conflict

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert os.path.isdir(ROOT), f"ROOT not found: {ROOT}"
for p in (CKPT_SHARED_PATH, CKPT_BASE_PATH, CKPT_INDEP_PATH):
    if not os.path.isfile(p): raise FileNotFoundError(p)

# 1) Load SHARED (to get K_shared & label space), then BASE with that K, then INDEP
shared_model, K_shared, num_labels_s, supercats_ckpt = load_shared_ckpt(CKPT_SHARED_PATH, device)
base_model,   K_base,   num_labels_b = load_baseline_ckpt(CKPT_BASE_PATH, device, num_parts=K_shared, num_labels_hint=num_labels_s)
indep_model,  K_indep,  num_labels_i = load_indep_ckpt (CKPT_INDEP_PATH,  device)

# 2) Annotator on CPU
annot_dev = torch.device("cpu")
vit_s = build_vits16_from_url(CKPT_URL_VITS16, device=annot_dev)
annot = DinoSelfAnnotator(vit_s, num_parts=int(K_shared), kmeans_iters=12, soft_temp=0.07).to(annot_dev)

# 3) Dataset & loader
ds = SharedMaskDataset(ROOT, annot, K_shared, out_size=IMG_SIZE, cache_dir="./mask_cache_shared")
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
super_list = supercats_ckpt if supercats_ckpt else ds.classes

# 4) If available, build spec with your function (present in your notebook)
if 'build_indep_spec' not in globals():
    raise RuntimeError("build_indep_spec not found in this notebook. Please run the cell that defines it.")
spec = build_indep_spec(super_list)
if spec.K_max != K_indep:
    print(f"[warn] spec tuples K={spec.K_max} != indep conv K={K_indep}. Proceeding, but check SUPER_TO_PARTS/PART_VOCAB.")

# 5) Evaluate all three
base_model.eval(); shared_model.eval(); indep_model.eval()
tot = 0; corr_b = corr_s = corr_i = 0
rows = []

with torch.inference_mode():
    for (x_rgb, m_shared, y, paths) in loader:
        x_rgb  = x_rgb.to(device)
        m_sh   = m_shared.to(device)
        y_dev  = y.to(device, dtype=torch.long)

        # BASE: x_rgb
        logits_b, _ = base_model(x_rgb)
        pred_b = logits_b.argmax(1)

        # SHARED: (x_rgb, m_shared)
        logits_s, _ = shared_model(x_rgb, m_sh)
        pred_s = logits_s.argmax(1)

        # INDEP: build m_ind from (m_shared, y) via spec
        m_ind = shared_to_indep_masks(m_sh, y_dev, spec, ds.classes)
        logits_i, _ = indep_model(x_rgb, m_ind)
        pred_i = logits_i.argmax(1)

        corr_b += (pred_b == y_dev).sum().item()
        corr_s += (pred_s == y_dev).sum().item()
        corr_i += (pred_i == y_dev).sum().item()
        tot    += y_dev.numel()

        for j in range(y_dev.numel()):
            rows.append([paths[j], int(y[j]), int(pred_b[j]), int(pred_s[j]), int(pred_i[j])])

acc_b = corr_b / max(tot,1)
acc_s = corr_s / max(tot,1)
acc_i = corr_i / max(tot,1)
print(f"[COMPARE] N={tot}")
print(f"  BASE   acc: {acc_b:.4f}  ({corr_b}/{tot})  (K_shared={K_base})")
print(f"  SHARED acc: {acc_s:.4f}  ({corr_s}/{tot})  (K_shared={K_shared})")
print(f"  INDEP  acc: {acc_i:.4f}  ({corr_i}/{tot})  (K_indep={K_indep})")

with open("predictions_compare.csv","w",newline="") as f:
    w = csv.writer(f)
    w.writerow(["path","gt_idx","pred_base","pred_shared","pred_indep"])
    w.writerows(rows)
print("Saved predictions_compare.csv")  

/home/xjzb2/miniconda3/envs/part_resnet/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[COMPARE] N=1280
  BASE   acc: 0.0852  (109/1280)  (K_shared=13)
  SHARED acc: 0.0625  (80/1280)  (K_shared=13)
  INDEP  acc: 0.1633  (209/1280)  (K_indep=42)
Saved predictions_compare.csv


In [37]:
ROOT = "model-vs-human/datasets/cue-conflict"  # the folder that contains airplane/, bear/, bicycle/, ...
SPLIT = None                    # <- important: no subfolder like "test" or "val"

test_ds = TripleViewPartTest(
    ROOT, SPLIT,
    annotator=annot,
    num_parts=annot.num_parts,
    out_size=IMG_SIZE,
    use_soft_masks=USE_SOFT_MASKS,
    cache_dir="./mask_cache_test"
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

NameError: name 'TripleViewPartTest' is not defined

In [ ]:
train_ps = PartStackedDataset(train_ds, annot, num_parts=8, out_size=224, build_mode="rgb_plus_mask", use_soft_masks=True)
val_ps   = PartStackedDataset(val_ds,   annot, num_parts=8, out_size=224, build_mode="rgb_plus_mask", use_soft_masks=True)

train_loader = torch.utils.data.DataLoader(train_ps, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_ps,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)


## On COCO

In [33]:

# === OVERRIDE: Compact independent-channel spec with K_max (max parts across classes) ===
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import torch, re

def _norm_str(s: str) -> str:
    return re.sub(r"[^a-z0-9]+","", s.lower())

@dataclass
class IndepSpec:
    """Compact independent-channel spec (K_max = max parts per class).
    For each class c, slot k uses shared channel `table[c,k]` or -1 (blank).
    """
    supers: List[str]                      # ordered superclass names
    part_to_shared: Dict[str, int]         # global part name -> shared idx (0..K_shared-1)
    class2len: Dict[str, int]              # number of parts this class has (<= K_max)
    table: torch.LongTensor                # shape (C, K_max), entries in [-1..K_shared-1]

    @property
    def K_shared(self) -> int:
        return int(max(self.part_to_shared.values()) + 1) if self.part_to_shared else 0

    @property
    def K_max(self) -> int:
        return int(self.table.shape[1]) if isinstance(self.table, torch.Tensor) else 0

def build_indep_spec(supercats: List[str]) -> IndepSpec:
    """Build compact spec with K_max channels (K_max = max parts across classes).
    Requires global PART_VOCAB, PART_TO_IDX, SUPER_TO_PARTS to be defined.
    """
    assert 'PART_VOCAB' in globals() and 'PART_TO_IDX' in globals() and 'SUPER_TO_PARTS' in globals(), \
        "PART_VOCAB, PART_TO_IDX, SUPER_TO_PARTS must be defined"
    part_to_shared = {p: int(PART_TO_IDX[p]) for p in PART_VOCAB}
    class2len = {s: len(SUPER_TO_PARTS.get(s, [])) for s in supercats}
    K_max = max(class2len.values()) if len(class2len)>0 else 0
    C = len(supercats)
    table = torch.full((C, K_max), -1, dtype=torch.long)
    for c, s in enumerate(supercats):
        parts = SUPER_TO_PARTS.get(s, [])
        for k, p in enumerate(parts[:K_max]):
            if p not in part_to_shared:
                raise KeyError(f"Part '{p}' for class '{s}' not in PART_VOCAB")
            table[c, k] = part_to_shared[p]
    return IndepSpec(supercats, part_to_shared, class2len, table)

@torch.no_grad()
def pack_shared_to_indep_batch(m_shared: torch.Tensor, pres_shared: torch.Tensor,
                               super_idx: torch.Tensor, spec: IndepSpec,
                               super_list: List[str]):
    """Compact mapping: (B, K_shared, H, W) -> (B, K_max, H, W) using class-dependent slots.
    Fills trailing channels with zeros when class has < K_max parts.
    Also maps presence targets accordingly.
    Returns: (m_indep, y_indep)
    """
    B, K_shared, H, W = m_shared.shape
    assert K_shared == spec.K_shared, f"K_shared mismatch: {K_shared} vs {spec.K_shared}"
    # gather per-sample slot->shared indices from table
    idx = spec.table.index_select(0, super_idx.detach().cpu()).to(m_shared.device)  # (B, K_max)
    valid = (idx >= 0).unsqueeze(-1).unsqueeze(-1)                                  # (B, K_max, 1, 1)

    # masks: gather over channel dim
    idx_safe = idx.clamp_min(0).unsqueeze(-1).unsqueeze(-1).expand(-1, -1, H, W)    # (B,K_max,H,W)
    m_ind = m_shared.gather(1, idx_safe)                                            # (B,K_max,H,W)
    m_ind = torch.where(valid, m_ind, torch.zeros_like(m_ind))                      # zero invalid

    # presence: gather vectorially
    y_ind = pres_shared.gather(1, idx.clamp_min(0))                                  # (B,K_max)
    y_ind = torch.where(idx>=0, y_ind, torch.zeros_like(y_ind))
    return m_ind, y_ind

@torch.no_grad()
def indep_logits_to_shared(logits_ind: torch.Tensor, y: torch.Tensor,
                           spec: IndepSpec, super_list: List[str]):
    """Map independent-slot logits back to shared-part logits per sample.
    For sample b with class c, for each slot k with table[c,k]>=0, copy:
        out[b, table[c,k]] = logits_ind[b, k]
    Else fill with a very negative value.
    Returns: (B, K_shared)
    """
    B = logits_ind.size(0)
    P_shared = spec.K_shared
    out = torch.full((B, P_shared), fill_value=-1e4, device=logits_ind.device)
    idx = spec.table.index_select(0, y.detach().cpu()).to(logits_ind.device)  # (B, K_max)
    valid = (idx >= 0)
    for b in range(B):
        mask_b = valid[b]
        if mask_b.any():
            shared_ids = idx[b][mask_b]
            out[b, shared_ids] = logits_ind[b][mask_b]
    return out
